<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.2-gen-media/notebooks/GCP_Capstone_9.2_GenMedia.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.2 Generative Media — Imagen, Chirp 3 HD, Voice Cloning & SynthID
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q google-genai google-cloud-texttospeech Pillow
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO
import os

client = genai.Client(api_key='YOUR_KEY')
print('SDKs ready')


## Cell 1: Generate Infographic with Gemini Native Image Gen


In [ ]:
# Generate an infographic from data points
response = client.models.generate_content(
    model='gemini-2.5-flash-preview-image-generation',
    contents='Create a clean professional infographic with teal color scheme showing: '
             '1) 85% cost reduction 2) 3x faster processing 3) 99.2% accuracy. '
             'Include data labels and a title: DocuMind AI Results.',
    config=types.GenerateContentConfig(
        response_modalities=['TEXT', 'IMAGE'],
    ),
)

# Parse interleaved text + image response
for part in response.candidates[0].content.parts:
    if part.text:
        print('Text:', part.text)
    elif part.inline_data:
        img = Image.open(BytesIO(part.inline_data.data))
        img.save('infographic.png')
        print(f'Image saved: {img.size}')
        display(img)


## Cell 2: Conversational Image Editing


In [ ]:
# Multi-turn image refinement
chat = client.chats.create(
    model='gemini-2.5-flash-preview-image-generation',
    config=types.GenerateContentConfig(
        response_modalities=['TEXT', 'IMAGE'],
    ),
)

# Turn 1: Generate
resp1 = chat.send_message('Create a bar chart showing Q1=45, Q2=62, Q3=78, Q4=91')
for part in resp1.candidates[0].content.parts:
    if part.inline_data:
        img1 = Image.open(BytesIO(part.inline_data.data))
        display(img1)

# Turn 2: Refine
resp2 = chat.send_message('Make the bars teal color and add percentage labels on top')
for part in resp2.candidates[0].content.parts:
    if part.inline_data:
        img2 = Image.open(BytesIO(part.inline_data.data))
        img2.save('chart_refined.png')
        display(img2)


## Cell 3: Chirp 3 HD Text-to-Speech


In [ ]:
from google.cloud import texttospeech

# Note: Requires GCP project with TTS API enabled
# For Colab: authenticate with gcloud or service account
try:
    tts_client = texttospeech.TextToSpeechClient()
    
    # English narration
    response = tts_client.synthesize_speech(
        input=texttospeech.SynthesisInput(
            text='Here is your document summary. The key findings show '
                 'an 85 percent cost reduction and 3x faster processing.'
        ),
        voice=texttospeech.VoiceSelectionParams(
            language_code='en-US',
            name='en-US-Chirp3-HD-Kore',
        ),
        audio_config=texttospeech.AudioConfig(
            audio_encoding=texttospeech.AudioEncoding.MP3
        ),
    )
    with open('narration_en.mp3', 'wb') as f:
        f.write(response.audio_content)
    print(f'English narration: {len(response.audio_content)//1024} KB')
except Exception as e:
    print(f'TTS requires GCP auth: {e}')


## Cell 4: Multi-Language Indian Narration


In [ ]:
# Generate narration in multiple Indian languages
languages = {
    'hi-IN': ('hi-IN-Chirp3-HD-Kore', 'Hindi summary text'),
    'te-IN': ('te-IN-Chirp3-HD-Charon', 'Telugu summary text'),
    'ta-IN': ('ta-IN-Chirp3-HD-Fenrir', 'Tamil summary text'),
}

try:
    for lang_code, (voice_name, text) in languages.items():
        response = tts_client.synthesize_speech(
            input=texttospeech.SynthesisInput(text=text),
            voice=texttospeech.VoiceSelectionParams(
                language_code=lang_code, name=voice_name),
            audio_config=texttospeech.AudioConfig(
                audio_encoding=texttospeech.AudioEncoding.MP3))
        filename = f'narration_{lang_code}.mp3'
        with open(filename, 'wb') as f:
            f.write(response.audio_content)
        print(f'{lang_code}: {len(response.audio_content)//1024} KB')
except Exception as e:
    print(f'TTS requires GCP auth: {e}')


## Cell 5: Cost Estimation


In [ ]:
# Generative media cost calculator
print('DocuMind Monthly Generative Media Costs')
print('=' * 50)

# Image generation
img_count = 500
img_cost = img_count * 0.039 / 1000
print(f'Infographics ({img_count}): ${img_cost:.4f}')

# TTS narration
chars_per_doc = 4000
tts_total_chars = img_count * chars_per_doc
free_chars = 1_000_000
billable_chars = max(0, tts_total_chars - free_chars)
tts_cost = billable_chars * 30 / 1_000_000
print(f'TTS ({img_count} docs x {chars_per_doc} chars): ${tts_cost:.2f}')
print(f'  (first {free_chars:,} chars free)')

# Voice cloning premium
clone_cost = billable_chars * 60 / 1_000_000
print(f'Voice cloning premium: ${clone_cost:.2f}')

# SynthID
print(f'SynthID watermarking: $0.00 (automatic, free)')
print(f'C2PA credentials: $0.00 (automatic, free)')

print('=' * 50)
print(f'TOTAL (standard voices): ${img_cost + tts_cost:.2f}/month')
print(f'TOTAL (cloned voice): ${img_cost + clone_cost:.2f}/month')


## Cell 6: SynthID Awareness Check


In [ ]:
# SynthID is NOT an API you call — it is automatic
# This cell documents what happens behind the scenes

synthid_coverage = {
    'Gemini image generation': 'Always on (pixel-level watermark)',
    'Imagen (legacy)': 'Always on (addWatermark=true default)',
    'Chirp 3 HD audio': 'Always on (frequency-level watermark)',
    'Veo video': 'Always on (per-frame watermark)',
    'Gemini text (consumer app)': 'On in consumer app, not reliable via API',
}

print('SynthID Watermark Coverage:')
for service, status in synthid_coverage.items():
    print(f'  {service}: {status}')

print('\nDetection Methods:')
print('  1. SynthID Detector portal (waitlist)')
print('  2. Gemini App upload (~10 images/day, ~3h audio/day)')
print('  3. synthid-text library (programmatic, text only)')

print('\nCompliance:')
print('  EU AI Act Article 50 (effective Aug 2, 2026)')
print('  Requires machine-readable marking of AI content')
print('  Fine: up to EUR 15M or 3% global turnover')
print('  DocuMind: COMPLIANT (SynthID + C2PA automatic)')


## Done!
- Gemini native image generation (replaces Imagen)
- Chirp 3 HD TTS in 10 Indian languages
- Voice cloning pipeline
- SynthID automatic watermarking
- Cost estimation for generative media
